# ReAct(Reason + Act)
- 인공지능(AI)이 문제를 해결할 때 추론(생각)과 행동(도구 사용)을 번갈아 수행하며 스스로 답을 찾아가는 AI 에이전트 설계 프레임워크
- 사용자 질문 → Agent가 확인 → 적절한 Tool을 선택 → Tool안에서 LLM이 답변 생성 → Agent가 최종 답변으로 정리

### 작동 방식
   (Thought-Action-Observation)ReAct 에이전트는 종료 조건이 충족될 때까지 4단계의 순환 루프를 반복합니다.
- Thought (생각): 현재 상황을 분석하고 다음에 무엇을 할지 계획(추론)합니다.
- Action (행동): 계획을 실행하기 위해 외부 도구(웹 검색, 계산기, API 등)를 호출합니다.
- Observation (관찰): 도구의 실행 결과(데이터)를 확인하고 새로운 정보를 습득합니다.
- Final Answer (최종 답변): 얻은 정보들을 조합하여 최종 결론을 도출합니다.

In [13]:
!pip install langchain
!pip install langchain-openai
!pip install openai
!pip install langchain-community
!pip install langchain-core
!pip install python-dotenv
!pip install langchain-chroma
!pip install sentence-transformers
!pip install langchain-classic

In [14]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
# ChatOpenAI: OpenAI의 GPT 모델(gpt-4o 등)을 대화형 인터페이스로 호출하는 클래스
# OpenAIEmbeddings: 텍스트를 AI가 이해할 수 있는 임베딩 벡터(숫자 배열)로 변환하는 클래스

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
# PromptTemplate: 일반적인 텍스트 프롬프트 템플릿을 생성 및 관리하는 클래스
# ChatPromptTemplate: 시스템, AI, 유저 역할이 구분된 대화형 프롬프트 템플릿을 만드는 클래스

from langchain_core.prompts import MessagesPlaceholder
# MessagesPlaceholder: 대화 기록(Memory)이나 동적인 메시지 배열이 들어갈 자리를 비워두는 틀

from langchain_core.output_parsers import StrOutputParser
# StrOutputParser: LLM의 출력 결과물(객체)에서 텍스트(문자열)만 깨끗하게 추출해 주는 파서

from langchain.tools import tool
# tool: 일반 파이썬 함수를 AI 에이전트가 인식하고 사용할 수 있는 전용 도구(Tool)로 바꾸는 데코레이터

from langchain_classic.agents import create_tool_calling_agent
# create_tool_calling_agent: LLM의 함수 호출(Function Calling) 기능을 기반으로 실행 계획을 세우는 에이전트를 생성하는 함수

from langchain_classic.agents import AgentExecutor
# AgentExecutor: 생성된 에이전트와 도구들을 결합하여 실제로 루프를 돌리며 실행을 제어하는 환경

import os
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY가 없어요")

llm = ChatOpenAI(model="gpt-4.1-mini", temperature="0.5")

In [20]:
# LLM이 생성한 문자열 정리용 - 공통 유틸리티 생성
def clean_answer(t:str) -> str:
  t = str(t).strip()

  # 전체 문자열이 두 번 반복한 패턴인지 검사
  if len(t) % 2 == 0 and t[:len(t) // 2] == t[len(t)//2:]:
    t = t[:len(t) // 2].strip() # 절반으로 나눠 앞 절반만 취함

  # 문단 단위로 중복 제거 '\n\n' 기준으로 문단 나눠서 이미 본 문단은 제외
  out, seen = [], set()
  for p in t.split("\n\n"):
    p = p.strip()
    if p and p not in seen: # 비어있지 않고 중복되지 않은 문단만 추가
      seen.add(p)
      out.append(p)

  merged = " ".join(out)
  cleaned = merged.replace("**","").replace("*","") # 마크다운 기호 제거
  return cleaned.strip()

# 프롬프트를 받아 LLM을 호출하고, 결과 문자열을 정리
def run_llm(prompt:str) -> str:
  resp = llm.invoke(prompt)
  return clean_answer(getattr(resp, "content", resp)) # getattr : 개체의 속성을 취함

In [21]:
import inspect
# Tool을 작성
@tool # 현재 함수는 LangChain tool로 등록
def math_helper(question:str)-> str:
  """
  수학 문제 푸는 도우미 함수
  """
  prompt = (
      "너는 수학 풀이르 잘하는 수학고수야.\n"
      "아래 수학 문제를 단계별로 풀고, 마지막 줄에 정확하게 정답을 적어 줘 \n\n"
      f"문제:{question}\n"
      "풀이 :"
  )
  return run_llm(prompt)

@tool # 현재 함수는 LangChain tool로 등록
def code_helper(question:str)-> str:
  """
  코딩 소스 작성 도우미
  """
  prompt = (
      "너는 10년 이상 경력을 가진 전문 프로그래머야.\n"
      "아래 요청에 대해 1)간단한 설명 2)예제 코드 3) 가장 중요한 포인트 설명 순서로 답해 줘 \n\n"
      f"요청:{question}\n"
      "답변:"
  )
  return run_llm(prompt)

@tool # 현재 함수는 LangChain tool로 등록
def general_helper(question:str)-> str:
  """
  일반 개념 / 이론을 이해하기 쉽게 5문장 이내로 설명하는 도우미
  """
  prompt = (
      "너는 친절한 AI 도우미야.\n"
      "아래 질문에 대해 5분장 이내로 답해줘 \n\n"
      f"요청:{question}\n"
      "답변:"
  )
  return run_llm(prompt)

tools = [math_helper, code_helper, general_helper] #  Agent로 넘겨줄 도구 목록

In [22]:
# Agent

prompt = ChatPromptTemplate.from_messages( # Agent용 전체 프롬프트 템플릿 정의 - 최신기술
    [
        (
            "system", # Agent역할, Tool 사용 규칙
            "너는 사용자의 질문을 보고 적절한 Tool을 선택하는 에이전트야.\n"
            "- 수식, 계산, 사칙연산, +, -, *, / 등이 보이면 math_helper를 사용해 \n"
            "- 함수, 클래스, 프로그램, 알고리즘, 파이썬/python/자바/java 등이 보이면 code_helper를 사용해 \n"
            "- 그 외에 일반적인 질문인 경우엔 general_helper 사용해 \n"
            "하지만 툴에서 반환된 텍스트를 그대로 복사해서 출력하지 말고"
            "Tool에 내용을 참고해 최종 답변을 한국어로 깔끔하게 작성해\n"
        ),

        # PlaceHolder (자리 표시자) : 이전 내용을 주입 -> 이전까지의 대화 내용을 기록
        # ex)
        #     사용자: 홍길동 학생의 출석상황을 알려줘  -> AI: 홍길동의 출석률은 95%야
        #     사용자: 그러면 수료가 가능해?  ☜ 얘만 보면 누구에 대한 질문인지 모름. 그래서 이전 대화 내용이 필요
        # 이전 대화 History를 넣기 위한 자리 정의 : 대화의 연속성과 문맥 유지가 목적
        MessagesPlaceholder(variable_name="chat_history"),
        (
            "human", # 현재 사용자가 입력한 질문이 들어가는 자리
            "{input}"
            # ex) askFunc(3+5=?) 했다고 하면 ("human",{input<- 3+5=?}) 이런식으로 들어감
        ),

        # Agent가 Tool 호출 등 중간 과정을 쌓는 내부 메모장 필요 -> 그래서 Agent가 스스로 결정을 내림.
        # agent_scratchpad : Agent Tool 호출 기록 : Agent 내부에서 reasoning + Tool call 기록
        MessagesPlaceholder(variable_name="agent_scratchpad"),

    ]
)

agnet = create_tool_calling_agent( # LLM + Tools + Prompt 연결
            # 사용자 질문을 보고, 어떤 툴을 사용할지 판단하고, 최종 답변을 만들 계획을 세움
            llm = llm,
            tools = tools,
            prompt = prompt,
)

agent_executor = AgentExecutor( # Agent를 실제로 실행하는 관리자
            agent=agnet,
            tools=tools,
            verbose = False # 내부 tool 호출 로그를 콘솔에 출력 여부
)

In [27]:
# 테스트
chat_history = [] # 대화 history 저장용 List
def askFunc(q:str): # 질의를 실행하는 함수
  print('\n질문 :', q)
  result = agent_executor.invoke(
      {
          "input":q, # PromptTemplate에의 {input}에 mapping
          "chat_history":chat_history, # 이전 턴(작업)들의 대화 히스토리(자동저장-agent_scratchpad)
      }
  )
  print("최종 답변 :",result['output'])
  chat_history.append(("human",q)) # History에 사용자(사람) 질문 저장
  chat_history.append(("ai",result['output'])) # History에 AI 답변 저장

q1 = "1에서 10까지의 정수에 대한 가우스정의를 사용해서 더하기는 얼마인가?"
askFunc(q1)
# q2 = "퀵정렬 알고리즘 이해용 파이썬 코드 작성해"
q2 = "위에 내용을 파이썬으로 작성하고, 다른 수학자의 정의로도 합을 구하는 코드도 만들어줘"
askFunc(q2)
# q3 = "일본 삿포로의 8월 평균 날씨와 여행하기 좋은 날짜 추천해줘"
q3 = "위에서본 각각의 수학자들에 대한 업적을 설명해줘"
askFunc(q3)


질문 : 1에서 10까지의 정수에 대한 가우스정의를 사용해서 더하기는 얼마인가?
최종 답변 : 1부터 10까지의 정수를 가우스 정리를 사용해 더하면, 합은 55입니다. 공식은 n(n+1)/2이며, 여기서 n=10을 대입하면 10*11/2=55가 됩니다.

질문 : 위에 내용을 파이썬으로 작성하고, 다른 수학자의 정의로도 합을 구하는 코드도 만들어줘
최종 답변 : 1부터 10까지 정수의 합을 구하는 두 가지 파이썬 코드를 소개합니다.

첫 번째는 가우스 공식 n(n+1)/2를 사용하는 방법입니다:
```python
n = 10
sum_gauss = n * (n + 1) // 2  # 정수 나눗셈 사용
print(f"1부터 {n}까지의 합은 {sum_gauss}입니다.")
```
이 방법은 반복문 없이 빠르게 합을 계산할 수 있습니다.

두 번째는 반복문을 사용하여 하나씩 더하는 방법입니다:
```python
total = 0
for i in range(1, 11):  # 1부터 10까지 반복
    total += i          # total에 i를 더함
print("1부터 10까지의 합:", total)
```
이 방법은 직관적이고 초보자도 이해하기 쉽습니다.

두 코드 모두 1부터 10까지의 합을 정확히 55로 계산합니다.

질문 : 위에서본 각각의 수학자들에 대한 업적을 설명해줘
최종 답변 : 가우스는 수학과 과학 전반에 걸쳐 혁신적인 업적을 남긴 천재 수학자로, 정수론, 복소수 해석, 최소자승법, 천문학의 궤도 계산, 자기학, 비유클리드 기하학 연구 등 여러 분야에서 중요한 기여를 했습니다. 그는 ‘수학의 왕자’로 불리며 현대 수학과 과학의 토대를 마련했습니다.

또 다른 유명한 수학자로는 아이작 뉴턴(미적분학과 운동법칙 발견), 레온하르트 오일러(그래프 이론과 수학 기호 도입), 앙리 푸앵카레(위상수학과 동역학계 이론 창시), 에밀리 뒤 샤틀레(뉴턴 역학 번역 및 해설) 등이 있습니다. 이들은 각자 수학과 과학 발전에 큰 영향을 끼쳤습니다.
